# LoanLens — Counterfactual Engine

SHAP answers:
> **Why did the model make this prediction?**

Counterfactual analysis answers a different question:
> **What realistic actionable change could potentially change the model's prediction?**

This notebook implements a constrained counterfactual engine for LoanLens. It keeps non-actionable variables such as `Credit_History` fixed and searches only realistic ranges learned from the training data.

At the end, we also scan the rejected test applications for a strong successful single-feature example that can be used as the project's portfolio/demo case.

## 1. Imports

The model was saved by `03_explainability.ipynb`.


In [ ]:
from pathlib import Path
from itertools import product

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


## 2. Locate the project and saved model

The notebook works whether VS Code launches it from the repository root or the `notebooks/` directory.


In [ ]:
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "loan_approval.csv"
model_path = project_root / "models" / "loanlens_logistic_pipeline.joblib"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

if not model_path.exists():
    raise FileNotFoundError(
        f"Saved model not found: {model_path}\n"
        "Run 03_explainability.ipynb first."
    )

df = pd.read_csv(data_path)
model = joblib.load(model_path)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Loaded model: {model_path}")


## 3. Recreate the same train/test split

We use the same split as the previous notebooks.

The test set remains unseen when defining realistic ranges for counterfactual search.


In [ ]:
X = df.drop(columns=["Loan_Status", "Loan_ID"])
y = df["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training applications:", len(X_train))
print("Test applications:", len(X_test))


## 4. Define the counterfactual policy

A counterfactual engine should not be allowed to modify every feature.

In particular, we deliberately keep `Credit_History` fixed.

This prevents the system from producing an unrealistic recommendation such as:

```text
Credit_History: 0 → 1
```

Instead, the engine searches variables that can reasonably be treated as application scenarios.


In [ ]:
ACTIONABLE_FEATURES = {
    "Applicant_Income": {
        "lower_quantile": 0.10,
        "upper_quantile": 0.90,
    },
    "Coapplicant_Income": {
        "lower_quantile": 0.10,
        "upper_quantile": 0.90,
    },
    "Loan_Amount": {
        "lower_quantile": 0.10,
        "upper_quantile": 0.90,
    },
    "Loan_Term": {
        "lower_quantile": 0.10,
        "upper_quantile": 0.90,
    },
}

FIXED_FEATURES = [
    column
    for column in X.columns
    if column not in ACTIONABLE_FEATURES
]

print("Actionable features:")
for feature in ACTIONABLE_FEATURES:
    print(" •", feature)

print("\nFixed features:")
for feature in FIXED_FEATURES:
    print(" •", feature)


## 5. Estimate realistic ranges

The search range is based on the **training data** rather than arbitrary numbers.

We use the 10th–90th percentile interval as a conservative approximation of a typical range.

This avoids searching obviously extreme values.


In [ ]:
bounds = {}

for feature, settings in ACTIONABLE_FEATURES.items():
    lower = X_train[feature].quantile(settings["lower_quantile"])
    upper = X_train[feature].quantile(settings["upper_quantile"])

    bounds[feature] = {
        "lower": float(lower),
        "upper": float(upper),
    }

bounds_df = pd.DataFrame(bounds).T

display(bounds_df)


## 6. Prediction helper

The model returns:
- predicted class
- probability of approval

We explicitly locate the `Approved` class rather than assuming a probability-column position.


In [ ]:
classifier = model.named_steps["model"]

approved_index = list(
    classifier.classes_
).index("Approved")


def predict_application(application: pd.DataFrame):
    prediction = model.predict(application)[0]

    approval_probability = model.predict_proba(
        application
    )[0, approved_index]

    return prediction, float(approval_probability)


## 7. Select a rejected application

We start with one rejected applicant.

This gives us a concrete case for the counterfactual engine.


In [ ]:
test_predictions = model.predict(X_test)

rejected_positions = np.where(
    test_predictions == "Rejected"
)[0]

if len(rejected_positions) == 0:
    raise RuntimeError("No rejected applications found.")

sample_position = int(rejected_positions[0])

original_application = X_test.iloc[[sample_position]].copy()

original_prediction, original_probability = (
    predict_application(original_application)
)

print("Original prediction:", original_prediction)
print(f"Original approval probability: {original_probability:.4%}")

display(
    original_application.T.rename(
        columns={
            original_application.index[0]: "Original Value"
        }
    )
)


## 8. Generate candidate values

We create realistic candidate values for each actionable feature.

`Loan_Term` is rounded to common month intervals.

Income and loan amount are rounded to practical increments.


In [ ]:
def candidate_values(
    feature,
    original_value,
    points=9
):
    lower = bounds[feature]["lower"]
    upper = bounds[feature]["upper"]

    values = np.linspace(
        lower,
        upper,
        points
    )

    if feature == "Loan_Term":
        values = np.round(values / 30) * 30

    elif feature in {
        "Applicant_Income",
        "Coapplicant_Income",
        "Loan_Amount",
    }:
        values = np.round(values / 1000) * 1000

    values = np.append(values, original_value)
    values = np.unique(values)

    return values


candidate_grid = {
    feature: candidate_values(
        feature,
        float(original_application.iloc[0][feature]),
        points=9,
    )
    for feature in ACTIONABLE_FEATURES
}

for feature, values in candidate_grid.items():
    print(f"{feature}: {values}")


## 9. Normalized change cost

Different features use different units.

Changing income by ₹10,000 and changing loan term by 30 months cannot be compared directly.

We therefore normalize each change using the training-data range.

Lower cost = smaller overall movement from the original application.


In [ ]:
def normalized_change_cost(
    original,
    candidate
):
    total = 0.0

    for feature in ACTIONABLE_FEATURES:
        lower = bounds[feature]["lower"]
        upper = bounds[feature]["upper"]

        scale = max(
            upper - lower,
            1e-9
        )

        total += abs(
            float(candidate[feature])
            - float(original[feature])
        ) / scale

    return total


# 10. Stage 1 — Single-feature counterfactual search

First we test each actionable variable independently.

This is both faster and more interpretable than immediately searching thousands of combinations.

For example:

```text
Loan Amount only
        ↓
Did prediction change?

Income only
        ↓
Did prediction change?
```


In [ ]:
single_feature_results = []

for feature in ACTIONABLE_FEATURES:

    original_value = float(
        original_application.iloc[0][feature]
    )

    for value in candidate_values(
        feature,
        original_value,
        points=15,
    ):

        if np.isclose(
            value,
            original_value
        ):
            continue

        candidate = original_application.copy()

        candidate.loc[
            candidate.index[0],
            feature
        ] = value

        prediction, probability = (
            predict_application(candidate)
        )

        cost = normalized_change_cost(
            original_application.iloc[0],
            candidate.iloc[0],
        )

        single_feature_results.append({
            "feature": feature,
            "new_value": value,
            "prediction": prediction,
            "approval_probability": probability,
            "change_cost": cost,
            "candidate": candidate,
        })

print(
    f"Evaluated {len(single_feature_results):,} "
    "single-feature scenarios."
)


## 11. Find successful single-feature counterfactuals


In [ ]:
single_feature_successes = [
    result
    for result in single_feature_results
    if result["prediction"] == "Approved"
]

single_feature_successes = sorted(
    single_feature_successes,
    key=lambda item: (
        item["change_cost"],
        -item["approval_probability"],
    )
)

print(
    "Successful single-feature counterfactuals:",
    len(single_feature_successes)
)


## 12. Display the best single-feature counterfactual

If one exists, this is our most interpretable scenario because only one input changes.


In [ ]:
if single_feature_successes:

    best_single = single_feature_successes[0]

    feature = best_single["feature"]

    print("Best single-feature counterfactual")
    print("Feature:", feature)
    print(
        f"Original value: "
        f"{original_application.iloc[0][feature]:,.2f}"
    )
    print(
        f"New value: "
        f"{best_single['new_value']:,.2f}"
    )
    print(
        f"Original approval probability: "
        f"{original_probability:.2%}"
    )
    print(
        f"New approval probability: "
        f"{best_single['approval_probability']:.2%}"
    )
    print(
        f"Change cost: "
        f"{best_single['change_cost']:.4f}"
    )

else:

    print(
        "No single-feature counterfactual was found."
    )


## 13. Stage 2 — Multi-feature search

If a single change cannot flip the decision, we can search combinations.

We keep the grid deliberately small so the notebook remains practical.

The goal is not brute force for its own sake. The goal is to find whether a **small combination of actionable changes** can alter the model prediction.


In [ ]:
MULTI_FEATURE_POINTS = 7

multi_grid = {
    feature: candidate_values(
        feature,
        float(original_application.iloc[0][feature]),
        points=MULTI_FEATURE_POINTS,
    )
    for feature in ACTIONABLE_FEATURES
}

multi_feature_results = []

for values in product(
    *[
        multi_grid[feature]
        for feature in ACTIONABLE_FEATURES
    ]
):

    candidate = original_application.copy()

    for feature, value in zip(
        ACTIONABLE_FEATURES,
        values
    ):
        candidate.loc[
            candidate.index[0],
            feature
        ] = value

    prediction, probability = (
        predict_application(candidate)
    )

    cost = normalized_change_cost(
        original_application.iloc[0],
        candidate.iloc[0],
    )

    multi_feature_results.append({
        "prediction": prediction,
        "approval_probability": probability,
        "change_cost": cost,
        "candidate": candidate,
    })

print(
    f"Evaluated {len(multi_feature_results):,} "
    "multi-feature scenarios."
)


## 14. Find successful multi-feature counterfactuals


In [ ]:
multi_feature_successes = [
    result
    for result in multi_feature_results
    if result["prediction"] == "Approved"
]

multi_feature_successes = sorted(
    multi_feature_successes,
    key=lambda item: (
        item["change_cost"],
        -item["approval_probability"],
    )
)

print(
    "Successful multi-feature counterfactuals:",
    len(multi_feature_successes)
)


## 15. Select the best counterfactual

We prefer:
1. a successful single-feature change, because it is easier to explain;
2. otherwise, the lowest-cost successful multi-feature scenario;
3. otherwise, we report that no actionable counterfactual was found.

This makes the output deterministic and interpretable.


In [ ]:
best_counterfactual = None
counterfactual_type = None

if single_feature_successes:

    best_counterfactual = single_feature_successes[0]
    counterfactual_type = "single-feature"

elif multi_feature_successes:

    best_counterfactual = multi_feature_successes[0]
    counterfactual_type = "multi-feature"


if best_counterfactual is not None:

    best_application = (
        best_counterfactual["candidate"]
    )

    print(
        f"Best counterfactual type: "
        f"{counterfactual_type}"
    )

    print(
        f"Approval probability: "
        f"{best_counterfactual['approval_probability']:.2%}"
    )

    print(
        f"Change cost: "
        f"{best_counterfactual['change_cost']:.4f}"
    )

else:

    print(
        "No successful actionable "
        "counterfactual was found."
    )


## 16. Compare the original application with the counterfactual

Only changed variables are displayed.


In [ ]:
if best_counterfactual is not None:

    comparison = pd.DataFrame({
        "Original": original_application.iloc[0],
        "Counterfactual": best_application.iloc[0],
    })

    comparison["Changed"] = (
        comparison["Original"]
        != comparison["Counterfactual"]
    )

    display(
        comparison[comparison["Changed"]]
    )

else:

    print(
        "No counterfactual available for comparison."
    )


## 17. Find the closest scenario even when no flip exists

A counterfactual engine should not simply return `None`.

If no scenario reaches `Approved`, we find the scenario with the highest approval probability and report it as the **best tested scenario**.

This lets LoanLens distinguish:

```text
No flip found
```

from:

```text
Nothing was searched
```


In [ ]:
best_overall = sorted(
    multi_feature_results,
    key=lambda item: (
        -item["approval_probability"],
        item["change_cost"],
    )
)[0]

best_overall_application = (
    best_overall["candidate"]
)

print(
    f"Original approval probability: "
    f"{original_probability:.2%}"
)

print(
    f"Highest approval probability found: "
    f"{best_overall['approval_probability']:.2%}"
)

print(
    f"Change cost: "
    f"{best_overall['change_cost']:.4f}"
)

best_overall_comparison = pd.DataFrame({
    "Original": original_application.iloc[0],
    "Best Scenario": best_overall_application.iloc[0],
})

best_overall_comparison["Changed"] = (
    best_overall_comparison["Original"]
    != best_overall_comparison["Best Scenario"]
)

display(
    best_overall_comparison[
        best_overall_comparison["Changed"]
    ]
)


## 18. Build a structured LoanLens counterfactual response

The frontend should receive clean structured data rather than sklearn objects or pandas dataframes.

This structure can later become part of the FastAPI response.


In [ ]:
if best_counterfactual is not None:

    original_row = original_application.iloc[0]
    counterfactual_row = best_application.iloc[0]

    changes = []

    for feature in ACTIONABLE_FEATURES:

        old_value = float(
            original_row[feature]
        )

        new_value = float(
            counterfactual_row[feature]
        )

        if not np.isclose(
            old_value,
            new_value
        ):
            changes.append({
                "feature": feature,
                "from": old_value,
                "to": new_value,
            })

    counterfactual_response = {
        "original_prediction": original_prediction,
        "original_approval_probability": round(
            original_probability,
            6,
        ),
        "counterfactual_found": True,
        "counterfactual_type": counterfactual_type,
        "counterfactual_approval_probability": round(
            best_counterfactual[
                "approval_probability"
            ],
            6,
        ),
        "change_cost": round(
            best_counterfactual[
                "change_cost"
            ],
            6,
        ),
        "changes": changes,
    }

else:

    counterfactual_response = {
        "original_prediction": original_prediction,
        "original_approval_probability": round(
            original_probability,
            6,
        ),
        "counterfactual_found": False,
        "best_tested_probability": round(
            best_overall[
                "approval_probability"
            ],
            6,
        ),
        "message": (
            "No realistic actionable scenario "
            "changed the model prediction within "
            "the tested ranges."
        ),
    }

counterfactual_response


## 19. Why "no counterfactual found" is a valid result

A weak system might always produce advice such as:

> "Increase your income."

That is not necessarily justified.

If `Credit_History` is the dominant model signal and it is fixed by our counterfactual policy, the engine may legitimately find no actionable path.

LoanLens should then report:

> **No realistic actionable scenario was found within the tested range.**

This is more honest than inventing a recommendation.


## 20. Important interpretation rule

A successful counterfactual means:

> "Under this trained model, this input change produced an Approved prediction."

It does **not** mean:

> "A real bank will approve the application."

The final frontend should therefore use language such as:

- "Model prediction changes"
- "Potential model scenario"
- "Within tested range"
- "Not a guarantee of approval"

rather than making a real-world approval promise.


## 21. LoanLens counterfactual architecture

We now have a useful decision-analysis pipeline:

```text
Applicant
    │
    ▼
Prediction
    │
    ├──────────────► Approval probability
    │
    ▼
SHAP
    │
    └──────────────► Why?
    │
    ▼
Counterfactual Engine
    │
    ├── Single-feature search
    │
    ├── Multi-feature search
    │
    └── Best tested scenario
    │
    ▼
Actionable explanation
```

This is the core differentiator from a simple loan approval classifier.


## 22. Select a strong portfolio/demo counterfactual

The single rejected application selected earlier is useful for testing the engine, but it may not be the best example for the final LoanLens demo.

We therefore scan the **rejected test applications** using the efficient single-feature search only. We rank successful cases by **probability gain** so we can identify an example where a realistic actionable change produces a clear model flip.

This scan does not modify `Credit_History`; it remains fixed because it is treated as non-actionable in LoanLens.

In [ ]:
# Efficient scan across rejected test applications
portfolio_successes = []

for position in rejected_positions:
    application = X_test.iloc[[position]].copy()
    _, base_probability = predict_application(application)

    for feature in ACTIONABLE_FEATURES:
        original_value = float(application.iloc[0][feature])

        for value in candidate_values(
            feature,
            original_value,
            points=15,
        ):
            if np.isclose(value, original_value):
                continue

            candidate = application.copy()
            candidate.loc[candidate.index[0], feature] = value

            prediction, probability = predict_application(candidate)

            if prediction == "Approved":
                cost = normalized_change_cost(
                    application.iloc[0],
                    candidate.iloc[0],
                )

                portfolio_successes.append({
                    "position": int(position),
                    "feature": feature,
                    "original_value": original_value,
                    "new_value": float(value),
                    "original_probability": base_probability,
                    "new_probability": probability,
                    "probability_gain": probability - base_probability,
                    "change_cost": cost,
                    "candidate": candidate,
                })

print(
    "Successful rejected applicants found:",
    len({case['position'] for case in portfolio_successes})
)
print(
    "Successful single-feature scenarios found:",
    len(portfolio_successes)
)

## 23. Rank the successful examples

A good demo case should show a meaningful probability improvement without pretending that the model's prediction is a guarantee of real-world approval.

We therefore show the successful cases ranked by probability gain.

In [ ]:
if portfolio_successes:
    portfolio_summary = pd.DataFrame([
        {
            "position": case["position"],
            "changed_feature": case["feature"],
            "original_value": case["original_value"],
            "new_value": case["new_value"],
            "original_probability": case["original_probability"],
            "new_probability": case["new_probability"],
            "probability_gain": case["probability_gain"],
            "change_cost": case["change_cost"],
        }
        for case in portfolio_successes
    ])

    portfolio_summary = (
        portfolio_summary
        .drop_duplicates(
            subset=[
                "position",
                "changed_feature",
                "new_value",
            ]
        )
        .sort_values(
            ["probability_gain", "change_cost"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    display(
        portfolio_summary.style.format({
            "original_value": "{:,.0f}",
            "new_value": "{:,.0f}",
            "original_probability": "{:.2%}",
            "new_probability": "{:.2%}",
            "probability_gain": "{:.2%}",
            "change_cost": "{:.4f}",
        })
    )
else:
    print("No successful single-feature examples were found.")

## 24. Inspect the strongest demo case

The strongest probability-gain case is selected automatically. We display the complete original application so the frontend can later reproduce the same scenario with real input fields.

In [ ]:
if portfolio_successes:
    hero_case = sorted(
        portfolio_successes,
        key=lambda item: (
            -item["probability_gain"],
            item["change_cost"],
        ),
    )[0]

    hero_application = X_test.iloc[[hero_case["position"]]].copy()
    hero_counterfactual = hero_case["candidate"]

    print("Selected portfolio/demo case")
    print("Test position:", hero_case["position"])
    print("Changed feature:", hero_case["feature"])
    print(
        f"{hero_case['feature']}: "
        f"{hero_case['original_value']:,.0f} → "
        f"{hero_case['new_value']:,.0f}"
    )
    print(
        f"Approval probability: "
        f"{hero_case['original_probability']:.2%} → "
        f"{hero_case['new_probability']:.2%}"
    )
    print(
        f"Probability gain: "
        f"{hero_case['probability_gain']:.2%}"
    )
    print(
        f"Change cost: {hero_case['change_cost']:.4f}"
    )

    print("\nOriginal application")
    display(
        hero_application.T.rename(
            columns={hero_application.index[0]: "Original Value"}
        )
    )

    print("Counterfactual application")
    hero_comparison = pd.DataFrame({
        "Original": hero_application.iloc[0],
        "Counterfactual": hero_counterfactual.iloc[0],
    })
    hero_comparison["Changed"] = (
        hero_comparison["Original"]
        != hero_comparison["Counterfactual"]
    )

    display(
        hero_comparison[hero_comparison["Changed"]]
    )
else:
    print("No portfolio/demo case is available.")

## 25. Interpretation of the demo case

The counterfactual should be communicated carefully:

- It is a **model-based scenario**, not a promise of real loan approval.
- The engine changes only variables defined as actionable by LoanLens.
- `Credit_History` remains fixed and is not presented as something the applicant can simply change.
- A probability crossing 50% means the trained classifier changes its predicted class under the tested scenario; it does **not** mean a bank must approve the loan.
- If no realistic scenario flips the prediction, LoanLens should report that honestly.

## 26. Next stage — FastAPI backend

The ML experimentation is now ready to become reusable backend logic.

Target endpoints:

```text
POST /predict
POST /explain
POST /counterfactual
POST /analyze
```

The strongest endpoint will eventually combine everything:

```text
POST /analyze
```

and return:

```text
prediction
approval_probability
top_explanatory_factors
counterfactual_found
counterfactual_changes
best_tested_scenario
```

The frontend team can consume this JSON without knowing anything about pandas, sklearn, SHAP, or the counterfactual search.